In [ ]:
import os
from google.colab import files

print("Please upload kaggle.json again...")
if os.path.exists('kaggle.json'):

    os.remove('kaggle.json')
uploaded = files.upload()

In [ ]:
if 'kaggle.json' in uploaded:
    print("File mil gayi! Downloading Dataset...")

    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d rahulbhalley/gopro-deblur

    print("Unzipping... (Thoda wait kar)")
    !unzip -q gopro-deblur.zip -d /content/dataset

    print("Dataset Wapis Aa Gaya! Ab training code run kar.")
else:
    print("File upload nahi hui. Dobara try kar.")


In [ ]:
!pip install segmentation_models_pytorch pytorch-msssim albumentations -q

print("All libraries installed!")

In [ ]:
# part 1 - till now ok ok

import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt
from tqdm import tqdm
import segmentation_models_pytorch as smp
import numpy as np
from pytorch_msssim import ssim
from scipy.ndimage import convolve
import random

# --- CONFIGURATION ---
DATASET_ROOT = '/content/dataset/gopro_deblur'
BATCH_SIZE   = 8
IMG_SIZE     = 256
EPOCHS       = 50
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_PATH    = 'deblur_best.pth'

print(f"Running on: {DEVICE}")

# 1. DATASET  
class GoProDataset(Dataset):
    def __init__(self, root_dir, img_size=256, augment=True):
        self.blur_paths  = []
        self.sharp_paths = []
        self.augment     = augment
        self.img_size    = img_size

        blur_dir  = os.path.join(root_dir, 'blur',  'images')
        sharp_dir = os.path.join(root_dir, 'sharp', 'images')

        if not os.path.exists(blur_dir):
            print(f"Path nahi mila: {blur_dir}")
            return

        for img in sorted(os.listdir(blur_dir)):
            if img.endswith(('.png', '.jpg', '.jpeg')):
                self.blur_paths.append(os.path.join(blur_dir, img))
                self.sharp_paths.append(os.path.join(sharp_dir, img))

        print(f"Found {len(self.blur_paths)} image pairs.")
        self.to_tensor = transforms.ToTensor()
        self.resize    = transforms.Resize((img_size, img_size))

    def __getitem__(self, index):
        blur_img  = np.array(Image.open(self.blur_paths[index]).convert('RGB'))
        sharp_img = np.array(Image.open(self.sharp_paths[index]).convert('RGB'))

        if self.augment:
            if random.random() > 0.5:
                blur_img  = np.fliplr(blur_img).copy()
                sharp_img = np.fliplr(sharp_img).copy()

            if random.random() > 0.5:   # was 0.7 — now more frequent
                blur_img  = np.flipud(blur_img).copy()
                sharp_img = np.flipud(sharp_img).copy()

            if random.random() > 0.5:
                k = random.choice([1, 2, 3])
                blur_img  = np.rot90(blur_img,  k).copy()
                sharp_img = np.rot90(sharp_img, k).copy()

            if random.random() > 0.4:
                h, w = blur_img.shape[:2]
                crop = random.randint(int(0.75 * h), h)
                x = random.randint(0, max(0, w - crop))
                y = random.randint(0, max(0, h - crop))
                blur_img  = blur_img[y:y+crop, x:x+crop]
                sharp_img = sharp_img[y:y+crop, x:x+crop]

            if random.random() < 0.6:
                pil_sharp = Image.fromarray(sharp_img)
                blur_type = random.choice(['gaussian', 'motion'])

                if blur_type == 'gaussian':
                    radius   = random.uniform(1.5, 4.0)
                    blur_img = np.array(pil_sharp.filter(ImageFilter.GaussianBlur(radius=radius)))

                else:
                    k      = random.choice([9, 13, 17, 21])
                    kernel = np.zeros((k, k))
                    if random.random() > 0.5:
                        kernel[k // 2, :] = 1.0 / k
                    else:
                        np.fill_diagonal(kernel, 1.0 / k)

                    blurred = np.zeros_like(sharp_img, dtype=np.float32)
                    for c in range(3):
                        blurred[:, :, c] = convolve(
                            sharp_img[:, :, c].astype(np.float32), kernel
                        )
                    blur_img = np.clip(blurred, 0, 255).astype(np.uint8)

        blur_t  = self.resize(self.to_tensor(Image.fromarray(blur_img)))
        sharp_t = self.resize(self.to_tensor(Image.fromarray(sharp_img)))
        return blur_t, sharp_t

    def __len__(self):
        return len(self.blur_paths)

# 2. LOSS 
class BetterLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg19(pretrained=True).features[:18].eval()
        for p in vgg.parameters():
            p.requires_grad = False
        self.vgg = vgg.to(DEVICE)
        self.l1  = nn.L1Loss()

    def forward(self, pred, target):
        l1_loss   = self.l1(pred, target)
        ssim_loss = 1 - ssim(pred, target, data_range=1.0, size_average=True)
        feat_loss = self.l1(self.vgg(pred), self.vgg(target))
        return (0.5 * l1_loss) + (0.4 * ssim_loss) + (0.1 * feat_loss)

# 3. MODEL  
model = smp.Unet(
    encoder_name           = "efficientnet-b3",
    encoder_weights        = "imagenet",
    in_channels            = 3,
    classes                = 3,
    activation             = None,
    decoder_attention_type = "scse",
    decoder_dropout        = 0.2   
).to(DEVICE)

print(f"Model ready | Params: {sum(p.numel() for p in model.parameters()):,}")

# 4. TRAINING LOOP
dataset = GoProDataset(DATASET_ROOT, augment=True)

if len(dataset) > 0:
    train_size = int(0.9 * len(dataset))
    val_size   = len(dataset) - train_size
    train_ds, val_ds = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=4,
                              shuffle=False, num_workers=2, pin_memory=True)

    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

    criterion = BetterLoss()

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=1e-6
    )

    best_val_loss            = float('inf')
    train_losses, val_losses = [], []

    print("Encoder frozen for first 10 epochs...")
    for param in model.encoder.parameters():
        param.requires_grad = False

    patience_counter      = 0
    EARLY_STOP_PATIENCE   = 10

    print("Training Started...")

    for epoch in range(EPOCHS):

        if epoch == 10:
            for param in model.encoder.parameters():
                param.requires_grad = True
            print("Encoder unfrozen at epoch 10!")

        # ---- TRAIN ----
        model.train()
        epoch_loss = 0
        loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{EPOCHS}]")

        for blur, sharp in loop:
            blur, sharp = blur.to(DEVICE), sharp.to(DEVICE)
            optimizer.zero_grad()
            output = model(blur)
            loss   = criterion(output, sharp)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()
            loop.set_postfix(loss=f"{loss.item():.4f}")

        avg_train = epoch_loss / len(train_loader)
        train_losses.append(avg_train)

        # ---- VALIDATE ----
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for blur, sharp in val_loader:
                blur, sharp = blur.to(DEVICE), sharp.to(DEVICE)
                val_loss += criterion(model(blur), sharp).item()

        avg_val = val_loss / len(val_loader)
        val_losses.append(avg_val)

        scheduler.step()

        print(f"Epoch {epoch+1:02d} | Train: {avg_train:.4f} | "
              f"Val: {avg_val:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

        # Save best model
        if avg_val < best_val_loss:
            best_val_loss    = avg_val
            patience_counter = 0
            torch.save(model.state_dict(), SAVE_PATH)
            print(f"  Best model saved! (val: {best_val_loss:.4f})")
        else:
            patience_counter += 1
            print(f"  No improvement ({patience_counter}/{EARLY_STOP_PATIENCE})")

            if patience_counter >= EARLY_STOP_PATIENCE:
                print(f"⏹ Early stopping triggered at epoch {epoch+1}!")
                break

        # Visualize every 5 epochs
        if (epoch + 1) % 5 == 0:
            with torch.no_grad():
                b, s = next(iter(val_loader))
                b = b[:1].to(DEVICE)
                s = s[:1].to(DEVICE)
                out = model(b)
                res = torch.cat([b, out, s], dim=3)
                res = res.squeeze().cpu().permute(1, 2, 0).clamp(0, 1).numpy()
                plt.figure(figsize=(14, 4))
                plt.imshow(res)
                plt.title(f"Epoch {epoch+1} | Input → Output → Target")
                plt.axis('off')
                plt.show()

    # Loss curve
    plt.figure(figsize=(10, 4))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses,   label='Val Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.title('Training Progress'); plt.legend(); plt.grid(True)
    plt.show()

    print(f"Done! Best val loss: {best_val_loss:.4f} — model saved to '{SAVE_PATH}'")
else:
    print("Dataset Missing.")


# 5. INFERENCE  
from google.colab import files

def predict(img_path=None):
    model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
    model.eval()

    if img_path is None:
        print("Upload your blurry image:")
        uploaded = files.upload()
        if not uploaded:
            print("No file uploaded.")
            return
        img_path = list(uploaded.keys())[0]

    original  = Image.open(img_path).convert('RGB')
    orig_size = original.size

    t   = transforms.Compose([transforms.Resize((256, 256)), transforms.ToTensor()])
    inp = t(original).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        out = model(inp)

    inp_show = inp.squeeze().cpu().permute(1, 2, 0).clamp(0, 1).numpy()
    out_show = out.squeeze().cpu().permute(1, 2, 0).clamp(0, 1).numpy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(inp_show); axes[0].set_title("Input (Blurry)",     fontsize=14); axes[0].axis('off')
    axes[1].imshow(out_show); axes[1].set_title("Output (Deblurred)", fontsize=14); axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    result = Image.fromarray((out_show * 255).astype(np.uint8))
    result = result.resize(orig_size, Image.LANCZOS)
    save_name = "deblurred_v2_" + os.path.basename(img_path)
    result.save(save_name)
    files.download(save_name)
    print(f"Saved: {save_name}")

predict()